# Tutorial 12: Universal Graph Neural Networks for Quantum Circuit Design

## The Problem with Tabular ML

In Tutorial 8, we trained a DNN on tabular data: each row mapped design parameters to Hamiltonian targets. This works — **but only for a fixed circuit topology**.

Add a second resonator? Remove the feedline? The input dimension changes and the model breaks.

## The Solution: Heterogeneous Graph ML on Geometric Embeddings

The **Universal GNN pipeline** replaces tabular features with **graph-structured geometric embeddings**:

1. **Design parameters** → `build_layout()` → **Shapely polygons**
2. Each component → **static embedding** = `param_sum ∥ geometric_moments ∥ shape_tensor`
3. Connections → **typed edges** with coupling type, center distances, overlap geometry
4. Full layout → **virtual hub node** connecting to all components
5. **HeteroConv GNN** learns context-aware predictions via typed message passing

Key: each component only predicts the Hamiltonian parameters that physically belong to it:
- Qubit → `qubit_freq`, `anharmonicity`
- Resonator → `cavity_freq`, `kappa`
- Edges → `g` (coupling strength)


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.manifold import TSNE
from sklearn.metrics import r2_score
from torch_geometric.loader import DataLoader

from squadds.ml.universal.geometry.layout import build_layout
from squadds.ml.universal.geometry.viz import plot_layout, plot_component
from squadds.ml.universal.features.node_encoder import (
    compute_static_embedding, get_polygon_for_component,
    static_embedding_dim, DEFAULT_SHAPE_RESOLUTION,
)
from squadds.ml.universal.features.moments import compute_moments, moment_names
from squadds.ml.universal.features.edge_extractor import EdgeFeatureExtractor, edge_feature_dim
from squadds.ml.universal.graph.netlist import CircuitNetlist, ComponentSpec, EdgeSpec
from squadds.ml.universal.graph.builder import UniversalGraphBuilder
from squadds.ml.universal.graph.virtual_hub import (
    _rasterize_in_bounds, compute_spatial_edge_features, spatial_edge_feature_dim,
    hub_embedding_dim,
)
from squadds.ml.universal.model.gat_model import (
    UniversalGNN, COMPONENT_TARGET_MASK, NODE_TARGET_NAMES, EDGE_TARGET_NAMES,
)
from squadds.ml.universal.trainer import UniversalTrainer

SHAPE_RES = DEFAULT_SHAPE_RESOLUTION
print(f"Shape resolution: {SHAPE_RES}x{SHAPE_RES}")
print(f"Node embedding dim: {static_embedding_dim(SHAPE_RES)}")
print(f"Edge feature dim: {edge_feature_dim(SHAPE_RES)}")
print(f"Spatial edge dim: {spatial_edge_feature_dim(SHAPE_RES)}")

seed = 42
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)


---
## 1. From Design Parameters to Physical Layout

Same dataset as Tutorial 8. Each row → `build_layout()` → Shapely polygons.


In [ ]:
df = pd.read_parquet("data/training_data.parquet").drop_duplicates().reset_index(drop=True)
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Design params: {['cross_length','cross_gap','claw_length','ground_spacing','coupling_length','total_length']}")
print(f"Targets: {['qubit_frequency_GHz','anharmonicity_MHz','cavity_frequency_GHz','kappa_kHz','g_MHz']}")
df.head(3)


In [ ]:
row = df.iloc[0]
lyt = build_layout(
    cross_length=row["cross_length"], cross_gap=row["cross_gap"],
    claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
    coupling_length=row["coupling_length"], total_length=row["total_length"],
)
fig = plot_layout(lyt)
plt.suptitle(f"Layout from Row #0: cross_length={row['cross_length']}, claw_length={row['claw_length']}", y=1.01)
plt.show()


---
## 2. Static Embedding: Polygon → Fixed-Size Vector

Each component polygon is converted to a **deterministic embedding**:

| Part | Description | Dim |
|---|---|---|
| **param_sum** | `sum(design_params.values())` — permutation-invariant | 1 |
| **moments** | area, perimeter, bbox_area, bbox_perimeter, fill_factor, compactness, aspect_ratio, circularity | 8 |
| **shape_tensor** | Scale-invariant rasterized mask (captures shape, not size) | R² |

This makes the embedding **universal** — same size regardless of component type.


In [ ]:
comp_names = ["qubit", "claw", "resonator", "feedline"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, name in enumerate(comp_names):
    comp_data = lyt[name]
    polygon = get_polygon_for_component(comp_data)
    params = comp_data.get("params", {})
    
    embedding = compute_static_embedding(polygon, params=params, shape_resolution=SHAPE_RES)
    moments = compute_moments(polygon)
    
    print(f"=== {name.upper()} ===")
    print(f"  params: {params}")
    print(f"  param_sum: {sum(params.values()) if params else 0:.1f}")
    for mname, mval in zip(moment_names(), moments):
        print(f"  {mname:20s}: {mval:12.2f}")
    print(f"  embedding dim: {len(embedding)}, norm: {np.linalg.norm(embedding):.2f}")
    print()
    
    shape_img = embedding[9:].reshape(SHAPE_RES, SHAPE_RES)
    axes[0, i].imshow(shape_img, cmap='viridis', interpolation='nearest')
    axes[0, i].set_title(f"{name.title()} Shape Tensor", fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].barh(moment_names(), moments, color='steelblue')
    axes[1, i].set_title(f"{name.title()} Moments", fontsize=10)
    axes[1, i].tick_params(labelsize=7)

plt.tight_layout()
plt.suptitle("Static Embeddings: Shape Tensors & Geometric Moments", fontsize=13, fontweight='bold', y=1.02)
plt.show()


### Embedding Space from Real SQuADDS Dataset

In [ ]:
dataset_embs, dataset_labels = [], []
for _, row in df.head(200).iterrows():
    ly = build_layout(cross_length=row["cross_length"], cross_gap=row["cross_gap"],
                     claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
                     coupling_length=row["coupling_length"], total_length=row["total_length"])
    for cn in ["qubit","claw","resonator","feedline"]:
        poly = get_polygon_for_component(ly[cn])
        emb = compute_static_embedding(poly, params=ly[cn].get("params",{}), shape_resolution=SHAPE_RES)
        dataset_embs.append(emb)
        dataset_labels.append(cn.title())

X_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(np.array(dataset_embs))
plt.figure(figsize=(10,7))
sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=dataset_labels, palette="deep", s=60, alpha=0.8)
plt.title("t-SNE of Component Embeddings (200 SQuADDS samples)")
plt.xlabel("Dim 1"); plt.ylabel("Dim 2"); plt.grid(alpha=0.3)
plt.show()


---
## 3. Heterogeneous Graph Assembly

The graph uses **typed nodes and edges** (PyG `HeteroData`):

| Node Type | Contains | Dim |
|---|---|---|
| `component` | Static embedding per component | param_sum + moments + shape = 265 |
| `virtual` | Full layout embedding + layer stack | ~269 |

| Edge Type | Between | Features |
|---|---|---|
| `physical` | component ↔ component | coupling_type + center dist + overlap geometry + overlap shape |
| `spatial_in` | component → virtual | rel_center + area_frac + perim_frac + masked shape |
| `spatial_out` | virtual → component | (same as spatial_in) |

Each component type only predicts **physically relevant targets**:

| Component | Predicts |
|---|---|
| TransmonCross | `qubit_freq`, `anharmonicity` |
| Claw | (nothing) |
| RouteMeander | `cavity_freq`, `kappa` |
| CoupledLineTee | (nothing) |
| physical edge | `g` (coupling) |


In [ ]:
netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator", component_type="RouteMeander"),
        ComponentSpec(name="feedline", component_type="CoupledLineTee"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator", coupling_type="galvanic"),
        EdgeSpec(src="resonator", dst="feedline", coupling_type="capacitive"),
    ],
)

builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")
data = builder.build(lyt, netlist, global_features={"dielectric_constant": 11.45, "substrate_thickness": 500})

print("=== HeteroData Graph ===")
print(data)
print()
print("Target mask (which component predicts which target):")
for i, (name, ctype) in enumerate(zip(data['component'].component_name, data['component'].component_type)):
    mask = data['component'].target_mask[i]
    targets = [t for t, m in zip(NODE_TARGET_NAMES, mask) if m]
    print(f"  {name:12s} ({ctype:16s}): {targets if targets else '(passive)'}")


In [ ]:
# Visualize edge overlap shape tensors
edge_extractor = EdgeFeatureExtractor(shape_resolution=SHAPE_RES)
edge_pairs = [("qubit","claw","capacitive"), ("claw","resonator","galvanic"), ("resonator","feedline","capacitive")]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (src, dst, ctype) in enumerate(edge_pairs):
    poly_a = get_polygon_for_component(lyt[src])
    poly_b = get_polygon_for_component(lyt[dst])
    feat = edge_extractor.extract(poly_a, poly_b, coupling_type=ctype)
    
    overlap_shape = feat[8:].reshape(SHAPE_RES, SHAPE_RES)
    dx, dy = feat[3], feat[4]
    ov_area = feat[5]
    
    print(f"Edge: {src} -> {dst} ({ctype})")
    print(f"  coupling one-hot: {feat[:3]}, center-to-center: dx={dx:.1f} dy={dy:.1f}, overlap_area={ov_area:.1f}")
    
    axes[i].imshow(overlap_shape, cmap='hot', interpolation='nearest')
    axes[i].set_title(f"{src} <-> {dst} ({ctype})", fontsize=10)
    axes[i].axis('off')

plt.suptitle("Physical Edge: Overlap Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Visualize hub-to-component masked shape tensors
from shapely.ops import unary_union
component_polygons = [get_polygon_for_component(lyt[n]) for n in ["qubit","claw","resonator","feedline"]]
layout_union = unary_union(component_polygons)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i, (name, poly) in enumerate(zip(["qubit","claw","resonator","feedline"], component_polygons)):
    masked = _rasterize_in_bounds(poly, layout_union.bounds, SHAPE_RES)
    axes[i].imshow(masked, cmap='Blues', interpolation='nearest')
    axes[i].set_title(f"Hub -> {name.title()}", fontsize=9)
    axes[i].axis('off')
    
    af = poly.area / layout_union.area
    pf = poly.length / layout_union.length
    print(f"Hub -> {name:12s}: area_frac={af:.4f}, perim_frac={pf:.4f}")

full_mask = _rasterize_in_bounds(layout_union, layout_union.bounds, SHAPE_RES)
axes[4].imshow(full_mask, cmap='Greens', interpolation='nearest')
axes[4].set_title("Hub Node (Full)", fontsize=9)
axes[4].axis('off')
plt.suptitle("Spatial Edges: Masked Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Building the Graph Dataset & Training

Each dataset row → HeteroData graph. Hamiltonian targets are assigned to the correct nodes/edges based on component type.


In [ ]:
N_SAMPLES = 5000  # Increase for production
df_sub = df.head(N_SAMPLES)

graph_dataset = []
builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")

print(f"Building {N_SAMPLES} graphs...")
for idx, row in df_sub.iterrows():
    lyt_i = build_layout(
        cross_length=row["cross_length"], cross_gap=row["cross_gap"],
        claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
        coupling_length=row["coupling_length"], total_length=row["total_length"],
    )
    data_i = builder.build(lyt_i, netlist, global_features={"dielectric_constant": 11.45})
    
    # Assign targets to the CORRECT nodes
    y = data_i["component"].y.clone()
    # Qubit (node 0): qubit_freq, anharmonicity
    y[0, 0] = row["qubit_frequency_GHz"]
    y[0, 1] = row["anharmonicity_MHz"] / 100.0  # scale
    # Resonator (node 2): cavity_freq, kappa
    y[2, 2] = row["cavity_frequency_GHz"]
    y[2, 3] = row["kappa_kHz"] / 100.0  # scale
    data_i["component"].y = y
    
    # Edge target: g on qubit-claw edge (indices 0,1 for both directions)
    y_edge = data_i["component", "physical", "component"].y.clone()
    y_edge[0, 0] = row["g_MHz"] / 100.0  # scale
    y_edge[1, 0] = row["g_MHz"] / 100.0
    data_i["component", "physical", "component"].y = y_edge
    
    graph_dataset.append(data_i)
    if (idx + 1) % 1000 == 0:
        print(f"  {idx+1}/{N_SAMPLES}")

print(f"Done: {len(graph_dataset)} graphs")


In [ ]:
split = int(0.85 * len(graph_dataset))
train_loader = DataLoader(graph_dataset[:split], batch_size=32, shuffle=True)
val_loader = DataLoader(graph_dataset[split:], batch_size=32)

sample = graph_dataset[0]
model = UniversalGNN(
    comp_dim=sample["component"].x.size(1),
    virt_dim=sample["virtual"].x.size(1),
    phys_edge_dim=sample["component", "physical", "component"].edge_attr.size(1),
    spat_edge_dim=sample["component", "spatial_in", "virtual"].edge_attr.size(1),
    hidden_dim=128, num_layers=3, num_heads=4, edge_hidden=32,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer = UniversalTrainer(model, learning_rate=1e-3, checkpoint_dir="checkpoints")
history = trainer.train_loop(train_loader, val_loader, epochs=500, patience=50)
trainer.load_checkpoint("best_model.pt")
print("Best model loaded.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set(xlabel="Epoch", ylabel="Total Loss", title="Training Curve"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history["train_node"], label="Train Node"); ax2.plot(history["val_node"], label="Val Node")
ax2.plot(history["train_edge"], label="Train Edge", ls='--'); ax2.plot(history["val_edge"], label="Val Edge", ls='--')
ax2.set(xlabel="Epoch", ylabel="Loss", title="Node vs Edge Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---\n## 5. Evaluation: Parity Plots

In [ ]:
model.eval()
all_yt, all_yp, all_yet, all_yep = [], [], [], []
with torch.no_grad():
    for batch in val_loader:
        out = model(batch)
        all_yt.append(batch["component"].y)
        all_yp.append(out["node_preds"])
        all_yet.append(batch["component", "physical", "component"].y)
        all_yep.append(out["edge_preds"])

yt = torch.cat(all_yt); yp = torch.cat(all_yp)
yet = torch.cat(all_yet); yep = torch.cat(all_yep)

# Node targets: [qubit_freq, anharmonicity, cavity_freq, kappa]
# Edge targets: [g]
plot_defs = [
    ("Qubit Freq (GHz)", 0, True, 1.0),
    ("Anharmonicity (MHz)", 1, True, 100.0),
    ("Cavity Freq (GHz)", 2, True, 1.0),
    ("Kappa (kHz)", 3, True, 100.0),
    ("Coupling g (MHz)", 0, False, 100.0),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, (name, idx, is_node, scale) in enumerate(plot_defs):
    ax = axes[i]
    if is_node:
        mask = ~torch.isnan(yt[:, idx])
        t = yt[mask, idx].numpy() * scale
        p = yp[mask, idx].numpy() * scale
    else:
        mask = ~torch.isnan(yet[:, idx])
        t = yet[mask, idx].numpy() * scale
        p = yep[mask, idx].numpy() * scale
    
    ax.scatter(t, p, alpha=0.4, s=10, color='crimson')
    if len(t) > 1:
        lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
        if lo != hi:
            ax.plot([lo, hi], [lo, hi], 'k--', lw=2)
            ax.text(0.05, 0.9, f"R2 = {r2_score(t, p):.3f}", transform=ax.transAxes, fontsize=12,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set(title=name, xlabel="True", ylabel="Predicted"); ax.grid(alpha=0.3)

axes[-1].axis('off')
plt.suptitle("Parity Plots", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6. Scale Invariance: The Holy Grail

The model was trained on **qubit-claw-resonator-feedline** graphs. Because it uses heterogeneous graph ML on geometric embeddings, it can handle **different topologies at inference time**.

### Case 1: Qubit-Claw Only

Remove the resonator and feedline. The model will predict `qubit_freq` and `anharmonicity` for the qubit node — and correctly produce **no** cavity/kappa predictions since no resonator is present.


In [ ]:
test_row = df.iloc[-1]
lyt_test = build_layout(
    cross_length=test_row["cross_length"], cross_gap=test_row["cross_gap"],
    claw_length=test_row["claw_length"], ground_spacing=test_row["ground_spacing"],
    coupling_length=test_row["coupling_length"], total_length=test_row["total_length"],
)

# Reduced netlist
reduced_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
    ],
    edges=[EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive")],
)

data_r = builder.build(lyt_test, reduced_netlist)
model.eval()
with torch.no_grad():
    out_r = model(data_r)

print("=== Case 1: Qubit-Claw Only ===")
print(f"Graph: {data_r['component'].x.size(0)} component nodes, {data_r['component','physical','component'].edge_index.size(1)} physical edges")
print()

for i, (name, ctype) in enumerate(zip(data_r['component'].component_name, data_r['component'].component_type)):
    mask = COMPONENT_TARGET_MASK[ctype]
    preds = out_r['node_preds'][i]
    scales = [1.0, 100.0, 1.0, 100.0]
    print(f"  {name} ({ctype}):")
    for j, (tname, m) in enumerate(zip(NODE_TARGET_NAMES, mask)):
        if m:
            print(f"    {tname}: {preds[j].item() * scales[j]:.3f}")

print(f"\nGround truth: qubit_freq={test_row['qubit_frequency_GHz']:.3f} GHz, anharmonicity={test_row['anharmonicity_MHz']:.2f} MHz")
print("Note: no cavity_freq/kappa predictions since there is no resonator in this topology.")

fig, ax = plt.subplots(figsize=(6, 6))
plot_component(lyt_test["qubit"], "qubit", ax=ax, show_etch=False)
plot_component(lyt_test["claw"], "claw", ax=ax, show_etch=False)
ax.autoscale_view()
ax.set_title("Case 1: Qubit-Claw Only", fontweight='bold')
plt.show()


### Case 2: Extended Topology (qubit-claw-resonator-feedline-feedline-resonator)

Now we **add** components. The model automatically predicts `cavity_freq` and `kappa` for the *second* resonator too — it was never trained on this topology!


In [ ]:
# For Case 2, we create an extended graph by reusing component geometries
# The key point: the GRAPH STRUCTURE is different, not the layout generation
lyt_ext = dict(lyt_test)
lyt_ext["feedline1"] = lyt_test["feedline"]
lyt_ext["resonator1"] = lyt_test["resonator"]
lyt_ext["feedline2"] = lyt_test["feedline"]  # same geometry, new graph node
lyt_ext["resonator2"] = lyt_test["resonator"]  # same geometry, new graph node
lyt_ext["design_params"] = lyt_test.get("design_params", {})

extended_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator1", component_type="RouteMeander"),
        ComponentSpec(name="feedline1", component_type="CoupledLineTee"),
        ComponentSpec(name="feedline2", component_type="CoupledLineTee"),
        ComponentSpec(name="resonator2", component_type="RouteMeander"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator1", coupling_type="galvanic"),
        EdgeSpec(src="resonator1", dst="feedline1", coupling_type="capacitive"),
        EdgeSpec(src="feedline1", dst="feedline2", coupling_type="galvanic"),
        EdgeSpec(src="feedline2", dst="resonator2", coupling_type="capacitive"),
    ],
)

data_ext = builder.build(lyt_ext, extended_netlist)

with torch.no_grad():
    out_ext = model(data_ext)

print("=== Case 2: Extended Topology (6 components) ===")
print(f"Graph: {data_ext['component'].x.size(0)} component nodes, "
      f"{data_ext['component','physical','component'].edge_index.size(1)} physical edges")
print()

scales = [1.0, 100.0, 1.0, 100.0]
for i, (name, ctype) in enumerate(zip(data_ext['component'].component_name, data_ext['component'].component_type)):
    mask = COMPONENT_TARGET_MASK[ctype]
    preds = out_ext['node_preds'][i]
    valid_targets = [(NODE_TARGET_NAMES[j], preds[j].item() * scales[j]) for j, m in enumerate(mask) if m]
    
    if valid_targets:
        tstr = ", ".join(f"{n}={v:.3f}" for n, v in valid_targets)
        print(f"  {name:14s} ({ctype:16s}): {tstr}")
    else:
        print(f"  {name:14s} ({ctype:16s}): (passive - no predictions)")

print()
print("Key observation: resonator2 automatically gets cavity_freq and kappa predictions!")
print("The model generalizes to unseen topologies through graph structure.")


---
## Summary

| Aspect | Tutorial 8 (Tabular DNN) | Tutorial 12 (Universal GNN) |
|---|---|---|
| Input | Fixed-size parameter vector | Heterogeneous graph of geometric embeddings |
| Targets | All predicted by one model | Per-component type (physically correct) |
| Topology | Fixed only | Any combination of components |
| Features | Raw numbers | Shape tensors + moments + graph structure |
| Inference on new topology | Impossible | Seamless |
